### Projeto de Arquitetura lakehouse com Databricks

#Ingestão de Dados Hidrológicos
### Hidrovia Paraguai-Paraná (Estação Assunción/Ladário)
Notebook responsável pela extraçào dos dados de níveis fluviais das estações Assunción e Ladárioda Hidrovia do Rio Paraguai. O monitoramento dessas estações é crítico para a logistica e transporte de commodities, segurança da navegação e estudos de impacto ambiental no Pantanal e na Bacia do Prata. 

Os dados são obtidos via web scraping da DMH (Dirección de Meteorología e Hidrología - Paraguay).

Destino:
- Parquet - Volume:
  - Asuncíon: `01_bronze/dmh/asuncion`
  - Ladário: `01_bronze/dhm/ladario`
- Tabela:
  - Asuncíon: `01_bronze.dmh_asuncion`
  - Ladário: `01_bronze.dmh_ladario`

Schema:

```python
root
 |-- cota: float (nullable = true)
 |-- data_reading: date (nullable = true)
 |-- month: integer (nullable = true)
 |-- year: integer (nullable = true)
 |-- ingestion_date: date (nullable = true)
```


In [0]:
## libs python

import logging
import requests
import time
import pandas as pd

from bs4 import BeautifulSoup

In [0]:
%python
## libs pyspark

from delta.tables import DeltaTable

from pyspark.sql import DataFrame
from pyspark.sql import functions as F
from pyspark.sql.types import StructType, StructField, FloatType, DateType, IntegerType, TimestampType

In [0]:
## constantes
BASE_URL = 'https://www.meteorologia.gov.py/nivel-rio/vermas_convencional.php'
ASUNCION_CODE = '2000086218'
LADARIO_CODE = '2000082001'

asuncion_parquet_path = '/Volumes/iguacu_lakehouse/01_bronze/dmh/asuncion'
ladario_parquet_path = '/Volumes/iguacu_lakehouse/01_bronze/dmh/ladario'
asuncion_table_name = 'iguacu_lakehouse.01_bronze.dmh_asuncion'
ladario_table_name = 'iguacu_lakehouse.01_bronze.dmh_ladario'

In [0]:
def setup_logging(name: str = "bronze_ingestion") -> logging.Logger:
    logger = logging.getLogger(name)
    logger.setLevel(logging.INFO)
    
    if not logger.handlers:
        handler = logging.StreamHandler()
        handler.setFormatter(logging.Formatter("%(asctime)s | %(levelname)s | %(message)s"))
        logger.addHandler(handler)
    
    return logger

logger = setup_logging()
     

In [0]:
## função utilitária
def __get_data_from_scraping__(url:str) -> pd.DataFrame:
    
    # Faz a requisição HTTP para a URL informada
    response = requests.get(url)
    
    # Verifica se a requisição foi bem sucedida (status 200)
    # Caso contrário, lança um erro
    if response.status_code == 200:
        soup = BeautifulSoup(response.text, 'html.parser')
        table = soup.find('table')
    
    
    # Converte a tabela HTML em um DataFrame do pandas
    # decimal e thousands ajudam a interpretar números corretamente
    df = pd.read_html(
        str(table),
        decimal='.',
        thousands=','
    )[0]
    
    # Retorna o DataFrame criado
    return df
    
    return None

In [0]:
# extract DMH Data
def extract_dmh_data(station_code:str, n_pages:int=11) -> DataFrame:
    
    schema = StructType([
        StructField('cota', FloatType(), True),
        StructField('data_reading', DateType(), True),
        StructField('month', IntegerType(), True),
        StructField('year', IntegerType(), True),
        StructField('ingestion_date', DateType(), True)
    ])

    raw_df = spark.createDataFrame([], schema)

    for page in range(1, n_pages):
        url = f'{BASE_URL}?code={station_code}&page={page}'
        page_data_df = __get_data_from_scraping__(url)
        
        if page_data_df is not None:
            data_df = spark.createDataFrame(
                page_data_df
            ).withColumn(
                'cota',
                F.regexp_replace(F.col('NIVEL DEL DÍA'), 'm', '').cast(FloatType())
            ).withColumn(
                'data_reading',
                F.to_date(F.col('FECHA'), 'dd-MM-yyyy')
            ).withColumn(
                'month',
                F.month(F.col('data_reading'))
            ).withColumn(
                'year',
                F.year(F.col('data_reading'))
            ).withColumn(
                'ingestion_date',
                F.current_date()
            ).select(
                'cota',
                'data_reading',
                'month',
                'year',
                'ingestion_date'
            )
            
        raw_df = raw_df.unionByName(data_df, allowMissingColumns=True)
        time.sleep(5)

    return raw_df
     

In [0]:
def merge_to_delta(df: DataFrame, table_name: str) -> None:
    # merger dos dados
    delta_table = DeltaTable.forName(spark, table_name)
    (
        delta_table.alias('target')
        .merge(
            df.alias('source'),
            'target.data_reading = source.data_reading'
        )
        .whenMatchedUpdateAll()
        .whenNotMatchedInsertAll()
        .execute()
    )

    logger.info(f'MERGE concluído em {table_name} com %s {df.count()}')

In [0]:
# extract Asuncion DMH Data
asuncion_df = extract_dmh_data(ASUNCION_CODE)
asuncion_df.orderBy('data_reading').limit(5).display()

In [0]:
##salvar arquivo parquet
(
    asuncion_df
        .write
        .partitionBy(['year', 'month'])
        .mode('append')
        .format('parquet')
        .save(asuncion_parquet_path)
)

In [0]:
merge_to_delta(asuncion_df, asuncion_table_name)

In [0]:
# extract Ladario DMH Data

ladario_df = extract_dmh_data(LADARIO_CODE)
ladario_df.orderBy('data_reading').limit(5).display()

In [0]:
##criar arquivo parquet
(
    ladario_df
        .write
        .partitionBy(['year', 'month'])
        .mode('append')
        .format('parquet')
        .save(ladario_parquet_path)
)

In [0]:
merge_to_delta(ladario_df, ladario_table_name)

In [0]:
%sql
SHOW TABLES IN iguacu_lakehouse.01_bronze;

In [0]:
%sql
SELECT 
  * 
FROM iguacu_lakehouse.01_bronze.dmh_asuncion;